# End-to-End Banking Data Pipeline
## Bronze → Silver → Gold

### Overview
This notebook combines the entire medallion architecture pipeline into a single executable notebook.

**Pipeline Flow:**
1. **Bronze Layer** — Generate raw mock banking data (customers, accounts, transactions, loan_applications)
2. **Silver Layer** — Clean, standardize, and validate the data
3. **Gold Layer** — Create aggregated analytical tables with the required features:
   - Monthly transaction aggregation by type
   - Loan approval demographics by province and income bracket
   - Customer aggregations including most common transaction type

**Catalog:** `sandbox_catalog`
**Schema:** `banking_details`

**Expected Runtime:** ~5-10 minutes depending on compute resources

---
## Imports & Configuration

In [0]:
import logging
import random
import re
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, DateType

# ── Logging setup ──────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("pipeline")

# ── Plot defaults ──────────────────────────────────────────
plt.rcParams["figure.dpi"]        = 110
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# ── Catalog / schema constants ─────────────────────────────
CATALOG = "sandbox_catalog"
SCHEMA  = "banking_details"

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

03:00:07  INFO  Imports complete. Catalog=sandbox_catalog  Schema=banking_details


In [0]:
# Create the schema (database) inside our catalog if it does not already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
log.info("Schema created/verified: %s.%s", CATALOG, SCHEMA)

01:20:40  INFO  Schema created/verified: sandbox_catalog.banking_details


---
## Helper Functions

In [0]:
# ── Bronze/Silver Helper Functions ────────────────────────────

def read_bronze(table_name: str):
    """Read a Bronze Delta table."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.read.table(full_name)
    log.info("Loaded Bronze <- %s  (%d rows)", full_name, df.count())
    return df

def write_silver(df, table_name: str, mode: str = "overwrite") -> None:
    """Write a cleaned Spark DataFrame as a Silver Delta table."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode(mode).saveAsTable(full_name)
    log.info("Written Silver -> %s  (%d rows)", full_name, df.count())

def null_report(df, label: str) -> None:
    """Print a per-column null count for the given DataFrame."""
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])
    print(f"\n── Null Report: {label} ──")
    null_counts.show()

def clean_column_names(df):
    """Lowercase all column names and replace spaces/special chars with underscores."""
    new_cols = [
        re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()
        for c in df.columns
    ]
    return df.toDF(*new_cols)

# ── Gold Helper Functions ────────────────────────────────────────

def read_table(table_name: str):
    """Read any Delta table from the active catalog/schema."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.read.table(full_name)
    log.info("Loaded %s (%d rows)", full_name, df.count())
    return df

def write_table(df, table_name: str, mode: str = "overwrite") -> None:
    """Write a Spark DataFrame as a Delta table."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode(mode).saveAsTable(full_name)
    log.info("Written %s (%d rows)", full_name, df.count())

log.info("Helper functions defined.")

03:00:15  INFO  Helper functions defined.


---
## BRONZE LAYER - Raw Data Generation

Generate mock banking data with intentional quality issues that will be cleaned in the Silver layer.

In [0]:
# Generate Customers table with mixed province formats
PROVINCE = [
    "Alberta", "AB", "alberta",
    "British Columbia", "BC", "british columbia",
    "Manitoba", "MB", "manitoba",
    "New Brunswick", "NB", "new brunswick",
    "Newfoundland and Labrador", "NL", "newfoundland and labrador",
    "Nova Scotia", "NS", "nova scotia",
    "Ontario", "ON", "ontario",
    "Prince Edward Island", "PE", "prince edward island",
    "Quebec", "QC", "quebec",
    "Saskatchewan", "SK", "saskatchewan"
]

customers_data = []
for i in range(1000):
    join_date = (datetime.now() - timedelta(days=random.randint(0, 365*10))).date()
    customers_data.append((
        i, 
        random.randint(18, 70), 
        random.choice(PROVINCE), 
        random.choice(["$0-$25K", "$25K-$50K", "$50K-$75K", "$75K-$100K", "$100K-$150K", "$150K-$200K", "$200K+"]), 
        join_date
    ))

spark.createDataFrame(
    customers_data, 
    ["customer_id", "age", "province", "income_bracket", "join_date"]
).write.mode("overwrite").saveAsTable("customers")

log.info("Bronze: customers table created")
display(spark.sql("SELECT * FROM customers LIMIT 5"))

01:21:04  INFO  Bronze: customers table created


customer_id,age,province,income_bracket,join_date
0,30,ON,$25K-$50K,2016-11-25
1,32,BC,$150K-$200K,2020-11-17
2,35,prince edward island,$200K+,2022-04-17
3,36,prince edward island,$50K-$75K,2022-08-04
4,50,Saskatchewan,$100K-$150K,2019-11-22


In [0]:
# Generate Accounts table with mixed case account types
ACCOUNT_TYPE = ['savings', 'CHECKINGS', 'Savings', 'Checkings']
STATUS = ['active', 'inactive', 'closed']
STATUS_WEIGHTS = [70, 20, 10]
CUSTOMER_IDS = list(range(1000))
START_DATE = (datetime.now() - timedelta(days=365*10)).date()
END_DATE = datetime.now().date()

accounts_data = []
for i in range(1500):
    customer_id = random.choice(CUSTOMER_IDS)
    random_days = random.randint(0, (END_DATE - START_DATE).days)
    open_date = (START_DATE + timedelta(days=random_days))
    accounts_data.append((
        i, 
        customer_id, 
        random.choice(ACCOUNT_TYPE),
        random.choice(STATUS), 
        open_date, 
        random.randint(1000, 1000000)
    ))

spark.createDataFrame(
    accounts_data, 
    ["account_id", "customer_id", "account_type", "status", "open_date", "balance"]
).write.mode("overwrite").saveAsTable("accounts")

log.info("Bronze: accounts table created")
display(spark.sql("SELECT * FROM accounts LIMIT 5"))

01:21:20  INFO  Bronze: accounts table created


account_id,customer_id,account_type,status,open_date,balance
750,835,Savings,closed,2018-06-30,982768
751,169,Savings,inactive,2016-11-11,520111
752,266,Savings,closed,2017-01-16,178677
753,159,savings,inactive,2022-03-05,583021
754,787,CHECKINGS,inactive,2022-03-27,611868


In [0]:
# Generate Transactions table with nulls and data issues
TRANSACTION_TYPE = ["deposit", "withdrawal", "transfer"]
TRANSACTION_WEIGHTS = [30, 50, 20]
MERCHANT_CATEGORIES = [
    "Groceries", "Dining", "Gas & Transport", 
    "Healthcare", "Entertainment", "Utilities", 
    "Online Shopping", "Travel", "ATM Withdrawal"
]
ACCOUNT_IDS = list(range(0, 1000))
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

transactions_data = []
for i in range(10000):
    account_id = random.choice(ACCOUNT_IDS)
    transaction_type = random.choices(TRANSACTION_TYPE, weights=TRANSACTION_WEIGHTS)[0]
    random_days = random.randint(0, (END_DATE - START_DATE).days)
    transaction_date = (START_DATE + timedelta(days=random_days)).strftime("%Y-%m-%d")
    
    # Add intentional nulls and negative amounts
    if random.random() < 0.03:
        amount = None
    elif transaction_type == "withdrawal" and random.random() < 0.05:
        amount = str(round(random.uniform(-500, -1), 2))
    else:
        amount = str(round(random.uniform(1, 5000), 2))
    
    # Merchant category only for withdrawals, sometimes null
    if transaction_type in ("deposit", "transfer"):
        merchant_category = None
    elif random.random() < 0.05:
        merchant_category = None
    else:
        merchant_category = random.choice(MERCHANT_CATEGORIES)
    
    transactions_data.append((
        i, account_id, transaction_type, transaction_date, amount, merchant_category
    ))

spark.createDataFrame(
    transactions_data, 
    ["transaction_id", "account_id", "transaction_type", "transaction_date", "amount", "merchant_category"]
).write.mode("overwrite").saveAsTable("transactions")

log.info("Bronze: transactions table created")
display(spark.sql("SELECT * FROM transactions LIMIT 5"))

01:21:22  INFO  Received command c on object id p0
01:21:27  INFO  Bronze: transactions table created


transaction_id,account_id,transaction_type,transaction_date,amount,merchant_category
1250,25,transfer,2024-12-17,131.17,null
1251,919,withdrawal,2024-01-07,548.94,Online Shopping
1252,249,deposit,2024-02-26,2490.08,null
1253,830,withdrawal,2024-10-20,2861.32,Groceries
1254,770,withdrawal,2024-03-18,142.96,Entertainment


In [0]:
# Generate Loan Applications table with nulls and outlier interest rates
LOAN_TYPE = ["mortgage", "auto", "personal"]
LOAN_STATUS = ["approved", "denied", "pending"]
STATUS_WEIGHTS = [50, 35, 15]

LOAN_AMOUNT_RANGES = {
    "mortgage": (100000, 900000),
    "auto": (5000, 80000),
    "personal": (1000, 50000)
}

INTEREST_RATE_RANGES = {
    "mortgage": (3.0, 7.0),
    "auto": (4.0, 12.0),
    "personal": (6.0, 25.0)
}

CUSTOMER_IDS = list(range(0, 1000))
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

loan_data = []
for i in range(600):
    customer_id = random.choice(CUSTOMER_IDS)
    loan_type = random.choice(LOAN_TYPE)
    random_days = random.randint(0, (END_DATE - START_DATE).days)
    application_date = (START_DATE + timedelta(days=random_days)).strftime("%Y-%m-%d")
    
    # Add some null amounts
    if random.random() < 0.05:
        amount_requested = None
    else:
        low, high = LOAN_AMOUNT_RANGES[loan_type]
        amount_requested = round(random.uniform(low, high), 2)
    
    status = random.choices(LOAN_STATUS, weights=STATUS_WEIGHTS)[0]
    
    # Add some outlier interest rates
    if random.random() < 0.03:
        interest_rate = round(random.uniform(100.0, 200.0), 2)
    else:
        low, high = INTEREST_RATE_RANGES[loan_type]
        interest_rate = round(random.uniform(low, high), 2)
    
    loan_data.append((
        i, customer_id, loan_type, amount_requested, status, application_date, interest_rate
    ))

spark.createDataFrame(
    loan_data,
    ["application_id", "customer_id", "loan_type", "amount_requested", "status", "application_date", "interest_rate"]
).write.mode("overwrite").saveAsTable("loan_applications")

log.info("Bronze: loan_applications table created")
display(spark.sql("SELECT * FROM loan_applications LIMIT 5"))

01:21:33  INFO  Bronze: loan_applications table created


application_id,customer_id,loan_type,amount_requested,status,application_date,interest_rate
0,147,auto,67157.32,approved,2024-01-17,8.18
1,326,mortgage,334012.37,denied,2024-08-02,4.72
2,70,personal,10811.29,denied,2024-11-06,23.72
3,356,personal,5226.93,approved,2024-05-03,6.79
4,124,mortgage,650156.28,denied,2024-09-12,6.63


---
## SILVER LAYER - Data Cleaning & Standardization

Clean and validate the Bronze data.

In [0]:
# Load Bronze customers
df_customers_bronze = read_bronze("customers")

# Province mapping to standardize all formats
province_map = {
    "alberta": "AB", "ab": "AB", "Alberta": "AB", "AB": "AB",
    "british columbia": "BC", "bc": "BC", "British Columbia": "BC", "BC": "BC",
    "manitoba": "MB", "mb": "MB", "Manitoba": "MB", "MB": "MB",
    "new brunswick": "NB", "nb": "NB", "New Brunswick": "NB", "NB": "NB",
    "newfoundland and labrador": "NL", "nl": "NL", "Newfoundland and Labrador": "NL", "NL": "NL",
    "nova scotia": "NS", "ns": "NS", "Nova Scotia": "NS", "NS": "NS",
    "ontario": "ON", "on": "ON", "Ontario": "ON", "ON": "ON",
    "prince edward island": "PE", "pe": "PE", "Prince Edward Island": "PE", "PE": "PE",
    "quebec": "QC", "qc": "QC", "Quebec": "QC", "QC": "QC",
    "saskatchewan": "SK", "sk": "SK", "Saskatchewan": "SK", "SK": "SK",
}

province_mapping_expr = F.create_map(
    *[item for pair in province_map.items() for item in (F.lit(pair[0]), F.lit(pair[1]))]
)

df_customers_clean = (
    df_customers_bronze
    .filter(F.col("customer_id").isNotNull())
    .withColumn("province", province_mapping_expr[F.col("province")])
    .filter(F.col("province").isNotNull())
    .filter(F.col("age").between(18, 100))
    .filter(F.col("income_bracket").isNotNull())
    .filter(F.col("join_date").isNotNull())
)

write_silver(df_customers_clean, "customers_silver")
log.info("Silver: customers_silver created")

01:21:36  INFO  Loaded Bronze <- sandbox_catalog.banking_details.customers  (1000 rows)
01:21:40  INFO  Written Silver -> sandbox_catalog.banking_details.customers_silver  (1000 rows)
01:21:40  INFO  Silver: customers_silver created


In [0]:
# Load and clean Bronze accounts
df_accounts_bronze = read_bronze("accounts")

VALID_STATUSES = ["active", "inactive", "closed"]

df_accounts_clean = (
    df_accounts_bronze
    .filter(F.col("account_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "account_type",
        F.when(
            F.lower(F.col("account_type")).isin("checkings", "chequing"), F.lit("chequing")
        ).when(
            F.lower(F.col("account_type")) == "savings", F.lit("savings")
        ).otherwise(None)
    )
    .filter(F.col("account_type").isNotNull())
    .withColumn("status", F.lower(F.col("status")))
    .filter(F.col("status").isin(VALID_STATUSES))
    .filter(F.col("balance") > 0)
    .filter(F.col("open_date").isNotNull())
)

write_silver(df_accounts_clean, "accounts_silver")
log.info("Silver: accounts_silver created")

01:21:42  INFO  Loaded Bronze <- sandbox_catalog.banking_details.accounts  (1500 rows)
01:21:47  INFO  Written Silver -> sandbox_catalog.banking_details.accounts_silver  (1500 rows)
01:21:47  INFO  Silver: accounts_silver created


In [0]:
# Load and clean Bronze transactions
df_transactions_bronze = read_bronze("transactions")

VALID_TRANSACTION_TYPES = ["deposit", "withdrawal", "transfer"]

df_transactions_clean = (
    df_transactions_bronze
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("account_id").isNotNull())
    .filter(F.col("amount").isNotNull())
    .withColumn("amount", F.col("amount").cast(DoubleType()))
    .filter(F.col("amount").isNotNull())
    .withColumn("amount", F.abs(F.col("amount")))
    .filter(F.col("amount") > 0)
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd"))
    .filter(F.col("transaction_date").isNotNull())
    .filter(F.col("transaction_type").isin(VALID_TRANSACTION_TYPES))
)

write_silver(df_transactions_clean, "transactions_silver")
log.info("Silver: transactions_silver created")

01:21:48  INFO  Loaded Bronze <- sandbox_catalog.banking_details.transactions  (10000 rows)
01:21:53  INFO  Written Silver -> sandbox_catalog.banking_details.transactions_silver  (9687 rows)
01:21:53  INFO  Silver: transactions_silver created


In [0]:
# Load and clean Bronze loan applications
df_loans_bronze = read_bronze("loan_applications")

VALID_LOAN_TYPES = ["mortgage", "auto", "personal"]
VALID_LOAN_STATUSES = ["approved", "denied", "pending"]
MAX_INTEREST_RATE = 30.0

df_loans_clean = (
    df_loans_bronze
    .filter(F.col("application_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("amount_requested").isNotNull())
    .filter(F.col("amount_requested") > 0)
    .filter(F.col("interest_rate") <= MAX_INTEREST_RATE)
    .filter(F.col("interest_rate").isNotNull())
    .filter(F.col("interest_rate") > 0)
    .withColumn("application_date", F.to_date(F.col("application_date"), "yyyy-MM-dd"))
    .filter(F.col("application_date").isNotNull())
    .filter(F.col("loan_type").isin(VALID_LOAN_TYPES))
    .filter(F.col("status").isin(VALID_LOAN_STATUSES))
)

write_silver(df_loans_clean, "loan_applications_silver")
log.info("Silver: loan_applications_silver created")

01:21:54  INFO  Loaded Bronze <- sandbox_catalog.banking_details.loan_applications  (600 rows)
01:21:58  INFO  Written Silver -> sandbox_catalog.banking_details.loan_applications_silver  (546 rows)
01:21:58  INFO  Silver: loan_applications_silver created


---
## GOLD LAYER - Aggregated Analytics

Create four Gold tables with required features:
1. **transactions_gold** — Enriched transactions with customer/account details
2. **customer_gold** — Customer aggregations with most common transaction type
3. **transaction_monthly_gold** — Monthly aggregations by transaction type
4. **loan_approval_demographics_gold** — Approval rates by province and income bracket

In [0]:
# Load all Silver tables
customers_s = read_table("customers_silver")
accounts_s = read_table("accounts_silver")
transactions_s = read_table("transactions_silver")
loan_applications_s = read_table("loan_applications_silver")

01:22:00  INFO  Loaded sandbox_catalog.banking_details.customers_silver (1000 rows)
01:22:01  INFO  Loaded sandbox_catalog.banking_details.accounts_silver (1500 rows)
01:22:02  INFO  Loaded sandbox_catalog.banking_details.transactions_silver (9687 rows)
01:22:03  INFO  Loaded sandbox_catalog.banking_details.loan_applications_silver (546 rows)


In [0]:
# Join transactions with accounts and customers
transactions_gold = (
    transactions_s
    .join(
        accounts_s.select("account_id", "customer_id", "account_type", "status", "open_date", "balance"),
        on="account_id",
        how="inner"
    )
    .join(
        customers_s.select("customer_id", "age", "province", "income_bracket", "join_date"),
        on="customer_id",
        how="inner"
    )
    .withColumn("transaction_year", F.year("transaction_date"))
    .withColumn("transaction_month", F.month("transaction_date"))
    .withColumn("transaction_quarter", F.quarter("transaction_date"))
    .withColumn("days_since_account_open", F.datediff(F.col("transaction_date"), F.col("open_date")))
    .withColumn("customer_tenure_days", F.datediff(F.col("transaction_date"), F.col("join_date")))
)

log.info("transactions_gold built with %d rows", transactions_gold.count())

01:22:06  INFO  transactions_gold built with 9687 rows


In [0]:
# Aggregate transaction behavior per customer
transaction_agg = (
    transactions_gold
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.sum("amount").alias("total_transaction_amount"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.max("transaction_date").alias("last_transaction_date"),
        F.sum(F.when(F.col("transaction_type") == "deposit", F.col("amount")).otherwise(0)).alias("total_deposits"),
        F.sum(F.when(F.col("transaction_type") == "withdrawal", F.col("amount")).otherwise(0)).alias("total_withdrawals"),
        F.sum(F.when(F.col("transaction_type") == "transfer", F.col("amount")).otherwise(0)).alias("total_transfers"),
        F.sum(F.when(F.col("transaction_type") == "deposit", 1).otherwise(0)).alias("count_deposits"),
        F.sum(F.when(F.col("transaction_type") == "withdrawal", 1).otherwise(0)).alias("count_withdrawals"),
        F.sum(F.when(F.col("transaction_type") == "transfer", 1).otherwise(0)).alias("count_transfers"),
    )
    .withColumn("total_transaction_amount", F.round("total_transaction_amount", 2))
    .withColumn("avg_transaction_amount", F.round("avg_transaction_amount", 2))
    .withColumn("total_deposits", F.round("total_deposits", 2))
    .withColumn("total_withdrawals", F.round("total_withdrawals", 2))
    .withColumn("total_transfers", F.round("total_transfers", 2))
)

# REQUIREMENT 3: Find most common transaction type per customer
most_common_type = (
    transactions_gold
    .groupBy("customer_id", "transaction_type")
    .agg(F.count("*").alias("type_count"))
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.desc("type_count"))
        )
    )
    .filter(F.col("rank") == 1)
    .select("customer_id", F.col("transaction_type").alias("most_common_transaction_type"))
)

transaction_agg = transaction_agg.join(most_common_type, on="customer_id", how="left")

# Aggregate account information
account_agg = (
    accounts_s
    .groupBy("customer_id")
    .agg(
        F.count("account_id").alias("total_accounts"),
        F.sum("balance").alias("total_balance"),
        F.avg("balance").alias("avg_balance"),
        F.sum(F.when(F.col("account_type") == "chequing", 1).otherwise(0)).alias("count_chequing"),
        F.sum(F.when(F.col("account_type") == "savings", 1).otherwise(0)).alias("count_savings"),
        F.sum(F.when(F.col("status") == "active", 1).otherwise(0)).alias("active_accounts"),
    )
    .withColumn("total_balance", F.round("total_balance", 2))
    .withColumn("avg_balance", F.round("avg_balance", 2))
)

# Aggregate loan information
loan_agg = (
    loan_applications_s
    .groupBy("customer_id")
    .agg(
        F.count("application_id").alias("total_loan_applications"),
        F.sum("amount_requested").alias("total_amount_requested"),
        F.avg("interest_rate").alias("avg_interest_rate"),
        F.sum(F.when(F.col("status") == "approved", 1).otherwise(0)).alias("approved_loans"),
        F.sum(F.when(F.col("status") == "denied", 1).otherwise(0)).alias("denied_loans"),
        F.sum(F.when(F.col("status") == "pending", 1).otherwise(0)).alias("pending_loans"),
    )
    .withColumn("total_amount_requested", F.round("total_amount_requested", 2))
    .withColumn("avg_interest_rate", F.round("avg_interest_rate", 2))
)

# Join everything together
customer_gold = (
    customers_s
    .join(transaction_agg, on="customer_id", how="left")
    .join(account_agg, on="customer_id", how="left")
    .join(loan_agg, on="customer_id", how="left")
    .fillna(0, subset=[
        "total_transactions", "total_transaction_amount", "avg_transaction_amount",
        "total_deposits", "total_withdrawals", "total_transfers",
        "count_deposits", "count_withdrawals", "count_transfers",
        "total_accounts", "total_balance", "avg_balance",
        "count_chequing", "count_savings", "active_accounts",
        "total_loan_applications", "total_amount_requested", "avg_interest_rate",
        "approved_loans", "denied_loans", "pending_loans",
    ])
    .withColumn(
        "customer_segment",
        F.when(F.col("total_transaction_amount") >= 100000, F.lit("High Value"))
        .when(F.col("total_transaction_amount") >= 50000, F.lit("Medium Value"))
        .when(F.col("total_transaction_amount") >= 10000, F.lit("Low Value"))
        .otherwise(F.lit("Inactive"))
    )
)

log.info("customer_gold built with %d rows", customer_gold.count())

01:22:07  INFO  Received command c on object id p0
01:22:12  INFO  customer_gold built with 1000 rows


In [0]:
# REQUIREMENT 1: Monthly transaction aggregation by type for seasonal analysis
transaction_monthly_gold = (
    transactions_gold
    .withColumn("year_month", F.date_format(F.col("transaction_date"), "yyyy-MM"))
    .groupBy("year_month", "transaction_type")
    .agg(
        F.sum("amount").alias("total_volume"),
        F.count("transaction_id").alias("transaction_count"),
        F.avg("amount").alias("avg_amount"),
    )
    .withColumn("total_volume", F.round("total_volume", 2))
    .withColumn("avg_amount", F.round("avg_amount", 2))
    .orderBy("year_month", "transaction_type")
)

log.info("transaction_monthly_gold built with %d rows", transaction_monthly_gold.count())

01:22:15  INFO  transaction_monthly_gold built with 36 rows


In [0]:
# REQUIREMENT 2: Loan approval analysis by province and income bracket

# Join loans with customer demographics
loan_with_demographics = (
    loan_applications_s
    .join(
        customers_s.select("customer_id", "province", "income_bracket"),
        on="customer_id",
        how="inner"
    )
)

# Aggregate by province
loan_approval_by_province = (
    loan_with_demographics
    .groupBy("province")
    .agg(
        F.count("application_id").alias("total_applications"),
        F.sum(F.when(F.col("status") == "approved", 1).otherwise(0)).alias("approved_count"),
        F.sum(F.when(F.col("status") == "denied", 1).otherwise(0)).alias("denied_count"),
        F.avg("amount_requested").alias("avg_amount_requested"),
        F.avg(F.when(F.col("status") == "approved", F.col("amount_requested")).otherwise(None)).alias("avg_amount_approved"),
    )
    .withColumn("approval_rate", F.round((F.col("approved_count") / F.col("total_applications")) * 100, 2))
    .withColumn("avg_amount_requested", F.round("avg_amount_requested", 2))
    .withColumn("avg_amount_approved", F.round("avg_amount_approved", 2))
    .withColumn("dimension", F.lit("province"))
    .withColumnRenamed("province", "dimension_value")
)

# Aggregate by income bracket
loan_approval_by_income = (
    loan_with_demographics
    .groupBy("income_bracket")
    .agg(
        F.count("application_id").alias("total_applications"),
        F.sum(F.when(F.col("status") == "approved", 1).otherwise(0)).alias("approved_count"),
        F.sum(F.when(F.col("status") == "denied", 1).otherwise(0)).alias("denied_count"),
        F.avg("amount_requested").alias("avg_amount_requested"),
        F.avg(F.when(F.col("status") == "approved", F.col("amount_requested")).otherwise(None)).alias("avg_amount_approved"),
    )
    .withColumn("approval_rate", F.round((F.col("approved_count") / F.col("total_applications")) * 100, 2))
    .withColumn("avg_amount_requested", F.round("avg_amount_requested", 2))
    .withColumn("avg_amount_approved", F.round("avg_amount_approved", 2))
    .withColumn("dimension", F.lit("income_bracket"))
    .withColumnRenamed("income_bracket", "dimension_value")
)

# Union both aggregations
loan_approval_demographics_gold = loan_approval_by_province.union(loan_approval_by_income)

log.info("loan_approval_demographics_gold built with %d rows", loan_approval_demographics_gold.count())

01:22:17  INFO  loan_approval_demographics_gold built with 17 rows


In [0]:
# Write all four Gold tables
write_table(transactions_gold, "transactions_gold")

# Write customer_gold with schema overwrite to include new column
customer_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.customer_gold")
log.info("Written %s.%s.customer_gold (%d rows)", CATALOG, SCHEMA, customer_gold.count())

write_table(transaction_monthly_gold, "transaction_monthly_gold")
write_table(loan_approval_demographics_gold, "loan_approval_demographics_gold")

log.info("All Gold tables written successfully!")

01:24:58  INFO  Written sandbox_catalog.banking_details.transactions_gold (9687 rows)
01:25:05  INFO  Written sandbox_catalog.banking_details.customer_gold (1000 rows)
01:25:12  INFO  Written sandbox_catalog.banking_details.transaction_monthly_gold (36 rows)
01:25:17  INFO  Written sandbox_catalog.banking_details.loan_approval_demographics_gold (17 rows)
01:25:17  INFO  All Gold tables written successfully!


---
## Pipeline Verification

Verify that all tables were created successfully.

In [0]:
# List all tables in the schema
tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
tables_list = [row.tableName for row in tables_df.collect()]

print("\n" + "="*70)
print("  PIPELINE VERIFICATION SUMMARY")
print("="*70)

# Bronze tables
bronze_tables = ["customers", "accounts", "transactions", "loan_applications"]
print("\n[BRONZE LAYER]")
for table in bronze_tables:
    status = "✅" if table in tables_list else "❌"
    if table in tables_list:
        count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
        print(f"  {status} {table:<30} {count:>10,} rows")
    else:
        print(f"  {status} {table:<30} NOT FOUND")

# Silver tables
silver_tables = ["customers_silver", "accounts_silver", "transactions_silver", "loan_applications_silver"]
print("\n[SILVER LAYER]")
for table in silver_tables:
    status = "✅" if table in tables_list else "❌"
    if table in tables_list:
        count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
        print(f"  {status} {table:<30} {count:>10,} rows")
    else:
        print(f"  {status} {table:<30} NOT FOUND")

# Gold tables
gold_tables = ["transactions_gold", "customer_gold", "transaction_monthly_gold", "loan_approval_demographics_gold"]
print("\n[GOLD LAYER]")
for table in gold_tables:
    status = "✅" if table in tables_list else "❌"
    if table in tables_list:
        count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
        print(f"  {status} {table:<30} {count:>10,} rows")
    else:
        print(f"  {status} {table:<30} NOT FOUND")

print("\n" + "="*70)
print("\n✅ Pipeline completed successfully!")
print("\nAll three requirements satisfied:")
print("  1. Monthly transaction aggregation by type (transaction_monthly_gold)")
print("  2. Loan approval demographics by province & income (loan_approval_demographics_gold)")
print("  3. Customer aggregations with most common transaction type (customer_gold)")
print("\n" + "="*70)


  PIPELINE VERIFICATION SUMMARY

[BRONZE LAYER]
  ✅ customers                           1,000 rows
  ✅ accounts                            1,500 rows
  ✅ transactions                       10,000 rows
  ✅ loan_applications                     600 rows

[SILVER LAYER]
  ✅ customers_silver                    1,000 rows
  ✅ accounts_silver                     1,500 rows
  ✅ transactions_silver                 9,687 rows
  ✅ loan_applications_silver              546 rows

[GOLD LAYER]
  ✅ transactions_gold                   9,687 rows
  ✅ customer_gold                       1,000 rows
  ✅ transaction_monthly_gold               36 rows
  ✅ loan_approval_demographics_gold         17 rows


✅ Pipeline completed successfully!

All three requirements satisfied:
  1. Monthly transaction aggregation by type (transaction_monthly_gold)
  2. Loan approval demographics by province & income (loan_approval_demographics_gold)
  3. Customer aggregations with most common transaction type (customer_gold)


---
# 📊 DASHBOARD PRESENTATION

The following cells display the Gold layer tables ready for dashboard visualization.

In [0]:
# Load Gold tables for dashboard presentation
customer_gold_dash = read_table("customer_gold")
transactions_gold_dash = read_table("transactions_gold")
transaction_monthly_gold_dash = read_table("transaction_monthly_gold")
loan_approval_demographics_gold_dash = read_table("loan_approval_demographics_gold")

# Display key business metrics
print("\n" + "="*70)
print("  KEY BUSINESS METRICS")
print("="*70)

# Customer metrics
total_customers = customer_gold_dash.count()
active_customers = customer_gold_dash.filter(F.col("total_transactions") > 0).count()
high_value_customers = customer_gold_dash.filter(F.col("customer_segment") == "High Value").count()

print(f"\n📊 CUSTOMER OVERVIEW")
print(f"  Total Customers:        {total_customers:>10,}")
print(f"  Active Customers:       {active_customers:>10,}")
print(f"  High Value Customers:   {high_value_customers:>10,}")

# Transaction metrics
total_transactions = transactions_gold_dash.count()
total_volume = transactions_gold_dash.agg(F.sum("amount")).collect()[0][0]
avg_transaction = transactions_gold_dash.agg(F.avg("amount")).collect()[0][0]

print(f"\n💰 TRANSACTION OVERVIEW")
print(f"  Total Transactions:     {total_transactions:>10,}")
print(f"  Total Volume:          ${total_volume:>10,.2f}")
print(f"  Avg Transaction:       ${avg_transaction:>10,.2f}")

# Loan metrics
total_applications = loan_approval_demographics_gold_dash.agg(F.sum("total_applications")).collect()[0][0]
avg_approval_rate = loan_approval_demographics_gold_dash.filter(F.col("dimension") == "province").agg(F.avg("approval_rate")).collect()[0][0]

print(f"\n🏦 LOAN OVERVIEW")
print(f"  Total Applications:     {int(total_applications):>10,}")
print(f"  Avg Approval Rate:      {avg_approval_rate:>10.2f}%")

print("\n" + "="*70)

03:00:57  INFO  Loaded sandbox_catalog.banking_details.customer_gold (1000 rows)
03:00:59  INFO  Loaded sandbox_catalog.banking_details.transactions_gold (9719 rows)
03:01:00  INFO  Loaded sandbox_catalog.banking_details.transaction_monthly_gold (36 rows)
03:01:02  INFO  Loaded sandbox_catalog.banking_details.loan_approval_demographics_gold (17 rows)



  KEY BUSINESS METRICS

📊 CUSTOMER OVERVIEW
  Total Customers:             1,000
  Active Customers:              641
  High Value Customers:            4

💰 TRANSACTION OVERVIEW
  Total Transactions:          9,719
  Total Volume:          $23,866,557.78
  Avg Transaction:       $  2,455.66

🏦 LOAN OVERVIEW
  Total Applications:          1,090
  Avg Approval Rate:           53.25%



## 📈 Requirement 1: Monthly Transaction Trends by Type

Seasonal pattern analysis showing transaction volume and count by month and type.

In [0]:
# Display monthly transaction trends
display(transaction_monthly_gold_dash.orderBy("year_month", "transaction_type"))

year_month,transaction_type,total_volume,transaction_count,avg_amount
2024-01,deposit,652437.85,259,2519.07
2024-01,transfer,370455.83,137,2704.06
2024-01,withdrawal,1002676.43,434,2310.31
2024-02,deposit,564451.41,222,2542.57
2024-02,transfer,321335.73,140,2295.26
2024-02,withdrawal,922698.0,385,2396.62
2024-03,deposit,617540.12,242,2551.82
2024-03,transfer,406319.76,168,2418.57
2024-03,withdrawal,1043678.13,427,2444.21
2024-04,deposit,620463.6,259,2395.61


## 👥 Requirement 2: Loan Approval Demographics

Approval rates and loan amounts by province and income bracket.

In [0]:
# Display loan approval by province
print("\n📍 LOAN APPROVAL BY PROVINCE\n")
display(
    loan_approval_demographics_gold_dash
    .filter(F.col("dimension") == "province")
    .select(
        F.col("dimension_value").alias("Province"),
        F.col("total_applications").alias("Total Applications"),
        F.col("approval_rate").alias("Approval Rate %"),
        F.col("avg_amount_requested").alias("Avg Amount Requested"),
        F.col("avg_amount_approved").alias("Avg Amount Approved")
    )
    .orderBy(F.desc("Approval Rate %"))
)


📍 LOAN APPROVAL BY PROVINCE



Province,Total Applications,Approval Rate %,Avg Amount Requested,Avg Amount Approved
NB,43,60.47,183509.63,133326.75
NL,57,59.65,169240.23,175179.27
PE,58,56.9,210428.59,232408.94
NS,51,56.86,280292.65,354886.83
BC,52,55.77,135103.73,161917.45
ON,59,52.54,215473.19,182217.49
QC,53,49.06,185104.04,160763.99
AB,59,47.46,164237.0,212515.4
SK,59,47.46,179443.06,183789.91
MB,54,46.3,227775.57,239166.19


In [0]:
# Display loan approval by income bracket
print("\n💵 LOAN APPROVAL BY INCOME BRACKET\n")
display(
    loan_approval_demographics_gold_dash
    .filter(F.col("dimension") == "income_bracket")
    .select(
        F.col("dimension_value").alias("Income Bracket"),
        F.col("total_applications").alias("Total Applications"),
        F.col("approval_rate").alias("Approval Rate %"),
        F.col("avg_amount_requested").alias("Avg Amount Requested"),
        F.col("avg_amount_approved").alias("Avg Amount Approved")
    )
    .orderBy(F.desc("Approval Rate %"))
)


💵 LOAN APPROVAL BY INCOME BRACKET



Income Bracket,Total Applications,Approval Rate %,Avg Amount Requested,Avg Amount Approved
$0-$25K,71,63.38,233185.15,257353.66
$200K+,79,55.7,168183.53,174602.79
$75K-$100K,68,52.94,202123.03,183816.69
$50K-$75K,67,52.24,187253.99,220811.33
$150K-$200K,77,51.95,183014.36,191717.38
$25K-$50K,99,49.49,207722.97,223654.27
$100K-$150K,84,47.62,183018.03,168668.25


## 🎯 Requirement 3: Customer Aggregations

Customer-level metrics including most common transaction type and segmentation.

In [0]:
# Display customer segmentation breakdown
print("\n🎯 CUSTOMER SEGMENTATION\n")
customer_segments = (
    customer_gold_dash
    .groupBy("customer_segment")
    .agg(
        F.count("*").alias("Customer Count"),
        F.sum("total_transaction_amount").alias("Total Volume"),
        F.avg("total_transaction_amount").alias("Avg Volume per Customer"),
        F.avg("total_transactions").alias("Avg Transactions per Customer")
    )
    .withColumn("Total Volume", F.round("Total Volume", 2))
    .withColumn("Avg Volume per Customer", F.round("Avg Volume per Customer", 2))
    .withColumn("Avg Transactions per Customer", F.round("Avg Transactions per Customer", 2))
    .orderBy(F.desc("Total Volume"))
)

display(customer_segments)


🎯 CUSTOMER SEGMENTATION



customer_segment,Customer Count,Total Volume,Avg Volume per Customer,Avg Transactions per Customer
Low Value,459,1.287149904E7,28042.48,11.67
Medium Value,156,1.035420446E7,66373.11,26.14
High Value,4,465723.72,116430.93,47.75
Inactive,381,175130.56,459.66,0.25


In [0]:
# Display most common transaction type distribution
print("\n🔄 MOST COMMON TRANSACTION TYPE BY CUSTOMER\n")
transaction_type_dist = (
    customer_gold_dash
    .filter(F.col("most_common_transaction_type").isNotNull())
    .groupBy("most_common_transaction_type")
    .agg(
        F.count("*").alias("Customer Count"),
        F.avg("total_transactions").alias("Avg Total Transactions"),
        F.avg("total_transaction_amount").alias("Avg Total Volume")
    )
    .withColumn("Avg Total Transactions", F.round("Avg Total Transactions", 2))
    .withColumn("Avg Total Volume", F.round("Avg Total Volume", 2))
    .orderBy(F.desc("Customer Count"))
)

display(transaction_type_dist)


🔄 MOST COMMON TRANSACTION TYPE BY CUSTOMER



most_common_transaction_type,Customer Count,Avg Total Transactions,Avg Total Volume
withdrawal,507,15.97,39278.97
deposit,102,12.93,31395.69
transfer,32,9.5,23430.07


In [0]:
# Display top 20 customers by transaction volume
print("\n⭐ TOP 20 CUSTOMERS BY TRANSACTION VOLUME\n")
top_customers = (
    customer_gold_dash
    .select(
        "customer_id",
        "age",
        "province",
        "income_bracket",
        "customer_segment",
        "total_transactions",
        "total_transaction_amount",
        "most_common_transaction_type",
        "total_balance",
        "total_accounts"
    )
    .orderBy(F.desc("total_transaction_amount"))
    .limit(20)
)

display(top_customers)


⭐ TOP 20 CUSTOMERS BY TRANSACTION VOLUME



customer_id,age,province,income_bracket,customer_segment,total_transactions,total_transaction_amount,most_common_transaction_type,total_balance,total_accounts
894,69,SK,$25K-$50K,High Value,55,132287.76,withdrawal,1601037,6
346,25,AB,$100K-$150K,High Value,54,122473.44,withdrawal,2927559,5
373,42,MB,$100K-$150K,High Value,42,107826.03,withdrawal,2476143,4
438,67,NL,$200K+,High Value,40,103136.49,withdrawal,3060586,5
111,54,SK,$200K+,Medium Value,39,99194.23,withdrawal,1762839,4
866,45,AB,$100K-$150K,Medium Value,41,99093.52,withdrawal,2016613,3
406,50,NS,$75K-$100K,Medium Value,34,97786.57,withdrawal,2446099,3
785,48,BC,$25K-$50K,Medium Value,37,97589.41,withdrawal,1176983,3
849,34,MB,$100K-$150K,Medium Value,40,97094.48,deposit,995344,4
85,69,BC,$150K-$200K,Medium Value,37,93795.21,withdrawal,1938968,4


In [0]:
# Display samples from each Gold table to verify data quality
print("\n=== TRANSACTIONS_GOLD Sample ===")
display(spark.table(f"{CATALOG}.{SCHEMA}.transactions_gold").limit(5))

print("\n=== CUSTOMER_GOLD Sample (showing most_common_transaction_type) ===")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.customer_gold")
    .select("customer_id", "age", "province", "total_transactions", "most_common_transaction_type", "customer_segment")
    .limit(5)
)

print("\n=== TRANSACTION_MONTHLY_GOLD Sample ===")
display(spark.table(f"{CATALOG}.{SCHEMA}.transaction_monthly_gold").limit(10))

print("\n=== LOAN_APPROVAL_DEMOGRAPHICS_GOLD Sample ===")
display(spark.table(f"{CATALOG}.{SCHEMA}.loan_approval_demographics_gold"))


=== TRANSACTIONS_GOLD Sample ===


customer_id,account_id,transaction_id,transaction_type,transaction_date,amount,merchant_category,account_type,status,open_date,balance,age,province,income_bracket,join_date,transaction_year,transaction_month,transaction_quarter,days_since_account_open,customer_tenure_days
305,972,0,withdrawal,2024-06-25,1352.82,null,savings,inactive,2019-02-18,626686,24,AB,$100K-$150K,2023-11-16,2024,6,2,1954,222
558,683,1,deposit,2024-10-10,2244.01,null,savings,closed,2023-02-24,557772,28,MB,$25K-$50K,2018-11-15,2024,10,4,594,2156
215,412,2,deposit,2024-02-07,3501.95,null,savings,inactive,2024-04-04,891131,38,NS,$25K-$50K,2022-10-12,2024,2,1,-57,483
501,878,3,deposit,2024-09-04,4958.8,null,chequing,closed,2016-07-16,410068,65,MB,$100K-$150K,2024-11-12,2024,9,3,2972,-69
963,112,4,deposit,2024-12-28,2917.91,null,savings,inactive,2021-03-17,494240,50,BC,$75K-$100K,2020-02-29,2024,12,4,1382,1764



=== CUSTOMER_GOLD Sample (showing most_common_transaction_type) ===


customer_id,age,province,total_transactions,most_common_transaction_type,customer_segment
0,30,ON,0,null,Inactive
1,32,BC,10,withdrawal,Low Value
2,35,PE,9,withdrawal,Low Value
3,36,PE,8,deposit,Low Value
4,50,SK,13,withdrawal,Low Value



=== TRANSACTION_MONTHLY_GOLD Sample ===


year_month,transaction_type,total_volume,transaction_count,avg_amount
2024-01,deposit,543666.3,222,2448.95
2024-01,transfer,505270.47,196,2577.91
2024-01,withdrawal,947101.93,405,2338.52
2024-02,deposit,589289.39,254,2320.04
2024-02,transfer,432432.07,157,2754.34
2024-02,withdrawal,902742.8,367,2459.79
2024-03,deposit,619969.89,251,2470.0
2024-03,transfer,471070.04,183,2574.15
2024-03,withdrawal,902942.46,377,2395.07
2024-04,deposit,559246.94,234,2389.94



=== LOAN_APPROVAL_DEMOGRAPHICS_GOLD Sample ===


dimension_value,total_applications,approved_count,denied_count,avg_amount_requested,avg_amount_approved,approval_rate,dimension
MB,40,13,17,173214.03,77127.73,32.5,province
AB,66,33,20,184799.48,204356.8,50.0,province
NS,48,25,12,179061.0,159375.64,52.08,province
NB,61,37,20,159744.94,137401.95,60.66,province
NL,58,31,17,178352.55,206531.19,53.45,province
BC,73,35,29,146025.74,137708.44,47.95,province
QC,51,25,18,270090.4,299017.95,49.02,province
ON,42,24,15,153273.17,173198.61,57.14,province
SK,52,27,14,209514.51,236569.11,51.92,province
PE,55,23,19,193572.99,228223.9,41.82,province
